# 🚀 STAIR: Structure-Aware Information Retriever - Kaggle T4 Fine-Tuning

This notebook fine-tunes **Llama-3.2-3B-Instruct** using **QLoRA (4-bit)** and **Unsloth** on a free **Kaggle Tesla T4 GPU** (16GB VRAM).

### STAIR Paper Hyperparameters:
- **Target Model:** `unsloth/llama-3.2-3b-instruct-bnb-4bit`
- **LoRA Rank (r):** 16, **LoRA Alpha:** 32, **Dropout:** 0.0
- **Target Modules:** `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`
- **Batch Size:** 2 per device × 4 gradient accumulation = 8 effective batch size
- **Learning Rate:** 2e-4 (cosine decay)
- **Max Context Window:** 4,096 tokens
- **Precision:** Mixed Precision FP16
- **Estimated Runtime:** ~15-20 minutes on Kaggle T4 GPU

## 1. Install Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth
!pip install trl datasets transformers peft bitsandbytes accelerate rank-bm25

## 2. Check Kaggle GPU Environment

In [ ]:
import torch
assert torch.cuda.is_available(), "Please enable GPU Accelerator in Kaggle settings (Settings -> Accelerator -> GPU T4 x 2 or P100)!"
print(f"✓ GPU Active: {torch.cuda.get_device_name(0)}")
print(f"✓ Available VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 3. Clone Repository or Prepare Data

In [ ]:
import os, json

# If running inside cloned repo, verify data files
train_path = "data/synthetic_qa_train.jsonl"
val_path = "data/synthetic_qa_val.jsonl"

if not os.path.exists(train_path):
    print("Creating synthetic data locally for training...")
    # Generate sample structure directly if data folder is not yet mounted
    os.makedirs("data", exist_ok=True)
    !python -c "from sample_pdf_processor import main; main()"
    !python -c "from src.2_generate_qa import main; main()"

from datasets import load_dataset
train_dataset = load_dataset("json", data_files=train_path, split="train")
val_dataset = load_dataset("json", data_files=val_path, split="train") if os.path.exists(val_path) else None

print(f"✓ Training examples loaded:   {len(train_dataset)}")
if val_dataset:
    print(f"✓ Validation examples loaded: {len(val_dataset)}")

## 4. Load Llama-3.2-3B with Unsloth 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096
MODEL_NAME = "unsloth/llama-3.2-3b-instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Apply LoRA with STAIR paper hyperparameters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)
print("✓ Model configured with LoRA r=16, alpha=32")

## 5. Configure Completion-Only Training

In [ ]:
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments

def formatting_prompts_func(examples):
    return [p + c + tokenizer.eos_token for p, c in zip(examples["prompt"], examples["completion"])]

# Train only on completion tokens (the leaf section name)
collator = DataCollatorForCompletionOnlyLM(
    response_template="### Section: leaf ",
    tokenizer=tokenizer
)

OUTPUT_DIR = "stair_lora_adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    fp16=True,
    logging_steps=5,
    eval_strategy="epoch" if val_dataset else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True if val_dataset else False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    formatting_func=formatting_prompts_func,
    data_collator=collator,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

## 6. Run Fine-Tuning

In [ ]:
print("🚀 Starting STAIR LoRA Fine-Tuning on Kaggle T4...")
trainer_stats = trainer.train()
print("✅ Fine-Tuning Complete!")
print(trainer_stats)

## 7. Save and Package LoRA Adapter for Download

In [ ]:
import zipfile

# Save model & tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

zip_filename = "stair_lora_adapter.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUTPUT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, start=os.path.dirname(OUTPUT_DIR))
            zipf.write(full_path, arcname)

print(f"📦 Adapter successfully packaged into: {zip_filename}")
print("You can now download 'stair_lora_adapter.zip' from the Kaggle Output file pane!")